# Hafta 14 — Uygulamada Makine Öğrenmesi ve Etik

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy shap fairlearn joblib

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Eğitilen bir modelin kaydedilip bir uygulamanın parçası haline getirilmesiyle iş bitmez. Zamanla insanların davranışları ve veriler değişir. Bu yüzden modelin başarısı düzenli olarak izlenmeli, gerektiğinde model yeniden eğitilmelidir.

Modelin kararları açıklanabilir olmalıdır. Örneğin bir kredi başvurusu reddedildiğinde "Kararı en çok mevcut borç miktarınız etkiledi." gibi anlaşılır bir gerekçe sunulabilmelidir.

Modeller adil de olmalıdır, çünkü geçmiş verilerdeki önyargıları da öğrenebilirler. Yıllarca çoğunlukla erkek çalışan işe almış bir şirketin verileriyle eğitilen bir model, kadın adaylara haksız biçimde düşük puan verebilir. Bu nedenle modelin farklı grupları nasıl etkilediği ölçülmeli ve bu konuda sorumluluk alınmalıdır.

---

## Bu Haftanın Hedefleri

- Uçtan uca bir ML pipeline'ı kurmak, modeli kaydetmek ve sunmak
- Permütasyon önemi ve SHAP ile model kararlarını açıklamak
- Adillik metriklerini hesaplamak
- Dönem projesini sunmak

---

## 1. Uçtan Uca Pipeline ve Model Kaydetme

Gerçek bir sistemde ön işleme ve model **tek bir nesne** olarak paketlenir. Böylece eğitimde ve kullanımda verinin birebir aynı şekilde işlenmesi garanti edilir.

**Problem:** Bir kişinin yıllık gelirinin 50.000 $'ı aşıp aşmadığını tahmin etmek (Adult veri seti).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

adult = fetch_openml("adult", version=2, as_frame=True)
X, y = adult.data, (adult.target == ">50K").astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

on_isleme = ColumnTransformer([
    ("say", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    ("kat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
     make_column_selector(dtype_exclude="number")),
])
model = make_pipeline(on_isleme, HistGradientBoostingClassifier(random_state=42)).fit(X_tr, y_tr)
print("Test ROC-AUC:", round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 4))

**Kaydetme ve yükleme:**

In [ ]:
import joblib

joblib.dump(model, "gelir_modeli.joblib")                 # ön işleme + model tek dosyada
yuklenen = joblib.load("gelir_modeli.joblib")
print(yuklenen.predict_proba(X_te.iloc[[0]]))

> Güvenmediğiniz kaynaklardan gelen `.joblib` / `pickle` dosyalarını yüklemeyin; bu dosyalar yüklenirken kod çalıştırabilir.

### Modeli bir servis olarak sunmak

Model bir web servisine dönüştürülerek başka uygulamaların kullanımına açılabilir. Aşağıdaki kod ayrı bir `servis.py` dosyasında çalışır (`uvicorn servis:app` komutuyla):

*Gösterim amaçlı kod (tek başına çalıştırılmaz):*

```python
from fastapi import FastAPI
import joblib, pandas as pd

app = FastAPI()
model = joblib.load("gelir_modeli.joblib")

@app.post("/tahmin")
def tahmin(basvuru: dict):
    olasilik = float(model.predict_proba(pd.DataFrame([basvuru]))[0, 1])
    return {"olasilik_50K_ustu": round(olasilik, 3)}
```

**İzleme:** Dağıtımdan sonra gelen verinin dağılımı eğitim verisinden uzaklaşabilir (**veri kayması**). Girdilerin dağılımı ve modelin başarısı düzenli olarak takip edilmelidir.

---

## 2. Açıklanabilirlik

### Permütasyon önemi

**Fikir:** Bir öznitelik önemliyse, değerlerini rastgele karıştırmak modelin başarısını düşürür.

```
1. Modelin skorunu ölç.
2. Bir sütunu karıştır, skoru yeniden ölç.
3. Önem = skordaki düşüş.
```

In [ ]:
from sklearn.inspection import permutation_importance

ornek = X_te.sample(2000, random_state=0)
sonuc = permutation_importance(model, ornek, y_te.loc[ornek.index], scoring="roc_auc",
                               n_repeats=3, random_state=0)
print(pd.Series(sonuc.importances_mean, index=X.columns).sort_values(ascending=False).head(8).round(4))

### SHAP: Tek bir kararı açıklamak

SHAP, oyun teorisindeki **adil paylaşım** fikrine dayanır. Bir tahmin, modelin ortalama tahmininden başlar; her özniteliğin katkısı eklenerek o kişinin tahminine ulaşılır.

**Elle örnek:** İki öznitelikli bir kredi risk modeli: Gelir (G) ve Borç (B).

| Bilinen öznitelikler | Modelin çıktısı |
|---|---|
| Hiçbiri (ortalama) | 50 |
| Yalnızca G | 70 |
| Yalnızca B | 60 |
| G ve B | 90 |

Her özniteliğin katkısı, iki olası ekleme sırasındaki katkılarının ortalamasıdır:

| Sıra | G'nin katkısı | B'nin katkısı |
|---|---|---|
| Önce G, sonra B | 70 − 50 = 20 | 90 − 70 = 20 |
| Önce B, sonra G | 90 − 60 = 30 | 60 − 50 = 10 |
| **Ortalama (SHAP)** | **25** | **15** |

**Kontrol:** 50 + 25 + 15 = 90. Tahmin, öznitelik katkılarıyla tam olarak açıklandı.

In [ ]:
try:
    # pip install shap
    import shap
    ozellikler = pd.DataFrame(model[0].transform(ornek), columns=model[0].get_feature_names_out())
    shap_degerleri = shap.TreeExplainer(model[-1])(ozellikler)
    shap.plots.waterfall(shap_degerleri[0], max_display=10)    # tek bir kişinin açıklaması
except ImportError as hata:
    print("Bu hücre için ek paket gerekli (pip install ...):", hata)

> SHAP, **modelin** nasıl karar verdiğini açıklar; gerçek dünyadaki neden-sonuç ilişkisini göstermez.

---

## 3. Yanlılık ve Adillik

### Yanlılık nereden gelir?

| Kaynak | Örnek |
|---|---|
| Geçmişteki ayrımcılık | Geçmiş işe alım kararlarından öğrenen model |
| Az temsil | Yüz tanıma verisinde bazı grupların çok az bulunması |
| Hatalı ölçüm | "Sağlık ihtiyacı" yerine "sağlık harcaması"nın hedef olarak kullanılması |

Cinsiyet veya etnik köken sütununu silmek yeterli değildir. Posta kodu, okul veya ilişki durumu gibi başka sütunlar bu bilgiyi dolaylı olarak taşıyabilir.

### Adillik metrikleri: Elle örnek

| | Grup A | Grup B |
|---|---|---|
| Başvuran | 100 | 100 |
| Gerçekte ödeyebilecek olan | 70 | 50 |
| Model onayladı | 60 | 30 |
| Ödeyebilecekler içinde onaylanan | 56 | 25 |

- **Demografik eşitlik** (onay oranları eşit mi?): A için 0.60, B için 0.30. Oran 0.30 / 0.60 = **0.50**. Sık kullanılan "%80 kuralına" göre 0.8'in altı olumsuz etki belirtisidir.
- **Fırsat eşitliği** (hak edenler eşit oranda onaylanıyor mu?): A'da 56/70 = **0.80**, B'de 25/50 = **0.50**. Krediyi ödeyebilecek B grubu üyelerinin yarısı reddediliyor.

Hangi adillik tanımının öncelikli olacağı teknik değil, **etik bir karardır** ve paydaşlarla birlikte verilmelidir.

In [ ]:
y_tahmin = model.predict(X_te)
for grup in X_te["sex"].unique():
    m = (X_te["sex"] == grup).values
    gercek, tahmin = y_te.values[m], y_tahmin[m]
    print(f"{grup:7s} onay oranı = {tahmin.mean():.3f}   fırsat (TPR) = {tahmin[gercek == 1].mean():.3f}")

### Diğer etik konular

- **Gizlilik:** Kişisel verilerin korunması (KVKK, GDPR).
- **Şeffaflık:** Modelin amacı, verisi ve sınırları belgelenmelidir (model kartları).
- **İnsan denetimi:** Sağlık, hukuk ve kredi gibi yüksek riskli kararlarda son söz insanda olmalıdır.

---

## 4. Laboratuvar: Dönem Projesi Sunumları

**Sunum yapısı (12 dakika):**

1. Problem ve motivasyon
2. Veri ve keşifsel analizden önemli bulgular
3. Yöntem: ön işleme, denenen modeller, doğrulama stratejisi
4. Sonuçlar: karşılaştırma tablosu ve hata analizi
5. Açıklanabilirlik, adillik ve sınırlamalar

**Kontrol listesi:**
- [ ] Test seti en başta ayrıldı ve yalnızca en sonda kullanıldı
- [ ] Ön işleme Pipeline içinde yapıldı (sızıntı yok)
- [ ] Basit bir başlangıç modeliyle (baseline) karşılaştırma yapıldı
- [ ] Probleme uygun metrik seçildi
- [ ] Modelin kararları açıklandı, sınırlamalar dürüstçe belirtildi

---

## 5. Özet

- Ön işleme ve model tek bir Pipeline olarak kaydedilir ve sunulur; dağıtımdan sonra izlenir.
- **Permütasyon önemi** genel olarak hangi özniteliklerin önemli olduğunu, **SHAP** ise tek bir kararın nasıl oluştuğunu açıklar.
- Adillik birden fazla biçimde tanımlanabilir; hangisinin seçileceği etik bir karardır.

## Kendinizi Sınayın

1. Bir modeli kaydederken neden ön işleme adımlarını da birlikte kaydederiz?
2. Bir işe alım modelinde kadın ve erkek adaylar için TPR değerleri 0.65 ve 0.85 çıkmış. Bu hangi adillik ilkesinin ihlalidir?
3. Dağıtımdan 6 ay sonra modelin başarısı düştü. Olası nedenler nelerdir?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.